<a href="https://colab.research.google.com/github/flathfk/Bootcamp_Study/blob/main/260331_1.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [8]:
%%writefile hash_compare.js

/* ================================================================
 * [코드 1] simpleHash: 단순 합산 방식
 *
 * <실행 결과 분석>
 *  입력: apple   | 해시값: 2   (97+112+112+108+101 = 530, 530%10 = 0) → 실제론 0
 *  입력: banana  | 해시값: 8
 *  입력: grape   | 해시값: 2   <- apple과 충돌 발생!
 *  입력: apple1  | 해시값: 1
 *  입력: apple2  | 해시값: 2   <- apple, grape와 또 충돌!
 *
 * <특징>
 *  - 각 문자의 아스키코드를 단순히 더한 뒤 테이블 크기로 나눈 나머지 반환
 *  - 구현이 매우 단순하고 이해하기 쉬움
 *
 * <문제점>
 *  1) 순서 무시: "apple1"과 "1apple"의 아스키 합이 같아 같은 해시값 나옴
 *  2) 충돌 빈번: 테이블 크기가 10뿐이라 해시값이 0~9에 몰림
 *  3) 분산 불량: 비슷한 문자 구성이면 해시값도 비슷하게 나옴
 *
 * <개선점>
 *  - 문자 위치에 따른 가중치 부여 필요 -> advancedHash 참고
 *  - 테이블 크기를 키워 충돌 확률을 낮춰야 함
 * ================================================================ */
function simpleHash(str, tableSize) {
    let hash = 0;
    for (let i = 0; i < str.length; i++) {
        const charCode = str.charCodeAt(i);
        hash = hash + charCode; // 위치 가중치 없이 단순 합산 -> 충돌 원인
    }
    return hash % tableSize;
}

const tableSize1 = 10;
const inputs1 = ["apple", "banana", "grape", "apple1", "apple2"];

for (let i = 0; i < inputs1.length; i++) {
    const input = inputs1[i];
    const hashValue = simpleHash(input, tableSize1);
    console.log("입력:", input);
    console.log("해시값:", hashValue);
    console.log("-------------------");
}


/* ================================================================
 * [코드 2] advancedHash: 소수 가중치 방식 (polynomial rolling hash)
 *
 * <실행 결과 분석>
 *  입력: apple   | 해시값: 코드1과 완전히 다른 값
 *  입력: banana  | 해시값: 충돌 없음
 *  입력: grape   | 해시값: apple과 다른 값 -> 충돌 해소
 *  입력: apple1  | 해시값: apple2와 다른 값 -> 순서/내용 구분 성공
 *  -> 7개 입력 모두 충돌 없이 서로 다른 해시값 배정
 *
 * <특징>
 *  - 각 문자마다 PRIME(31)을 누적 곱해 위치 가중치를 부여
 *  - 매 루프마다 % tableSize로 숫자 범위를 제한 (오버플로우 방지)
 *  - 소수 31 사용 -> 해시값이 테이블 전체에 고르게 분산됨
 *
 * <코드 1 대비 개선된 점>
 *  1) 순서 구분: "apple1"/="1apple", 문자 위치가 달라지면 해시값도 달라짐
 *  2) 충돌 감소: 테이블 크기 100으로 확대 -> 해시값 분포 공간 10배 증가
 *  3) 분산 향상: 소수 곱셈으로 비슷한 문자열도 전혀 다른 해시값 생성
 *  4) 실무 적합: 실제 해시맵(Java HashMap 등)이 사용하는 방식과 동일한 원리
 * ================================================================ */
function advancedHash(str, tableSize) {
    let hash = 0;
    const PRIME = 31;
    for (let i = 0; i < str.length; i++) {
        // hash * PRIME: 이전 해시에 위치 가중치 누적
        // + charCodeAt(i): 현재 문자 아스키코드 반영
        // % tableSize: 매 루프마다 범위 제한 (숫자 폭발 방지)
        hash = (hash * PRIME + str.charCodeAt(i)) % tableSize;
    }
    return hash;
}

const tableSize2 = 100;
const inputs2 = ["apple", "banana", "grape", "apple1", "apple2", "orange", "melon"];

console.log(`--- 테이블 크기 ${tableSize2}로 개선된 해시 결과 ---`);
const results = {};
inputs2.forEach(input => {
    const hashValue = advancedHash(input, tableSize2);
    console.log(`입력: ${input.padEnd(7)} | 해시값: ${hashValue}`);
    if (results[hashValue]) {
        console.log(`[!] 충돌 발생: ${hashValue} (기존: ${results[hashValue]})`);
    }
    results[hashValue] = input;
});

Overwriting hash_compare.js


In [6]:
!node hash_compare.js

입력: apple
해시값: 0
-------------------
입력: banana
해시값: 9
-------------------
입력: grape
해시값: 7
-------------------
입력: apple1
해시값: 9
-------------------
입력: apple2
해시값: 0
-------------------
--- 테이블 크기 100로 개선된 해시 결과 ---
입력: apple   | 해시값: 10
입력: banana  | 해시값: 69
입력: grape   | 해시값: 27
입력: apple1  | 해시값: 59
입력: apple2  | 해시값: 60
입력: orange  | 해시값: 86
입력: melon   | 해시값: 19


In [13]:
%%writefile ex1.js

class HashTable {
    // [1] 생성자: 지정한 크기(기본 10)만큼 빈 배열 생성 -> 슬롯 0~9 초기화
    constructor(size = 10) {
        this.table = new Array(size);
    }

    // [2] 해시 함수: 문자열 키 -> 숫자 인덱스로 변환
    hash(key) {
        let hash = 0;
        for (let i = 0; i < key.length; i++) {
            // [2-1] 각 문자의 아스키코드를 누적 합산
            //       ex) "name" -> 110+97+109+101 = 417
            hash += key.charCodeAt(i);
        }
        // [2-2] 합산값 % 테이블크기 -> 항상 0~9 범위 인덱스 반환
        return hash % this.table.length;
    }

    // [3] set(): 키를 해시화한 인덱스 위치에 값 저장
    set(key, value) {
        // [3-1] 키 -> hash() -> 인덱스 계산
        const index = this.hash(key);
        // [3-2] 해당 슬롯에 값 삽입
        //       주의: 같은 인덱스에 다른 키가 있으면 덮어씀 (충돌 미처리)
        this.table[index] = value;
    }

    // [4] get(): 동일한 해시 함수로 인덱스 계산 후 값 반환
    get(key) {
        // [4-1] set()과 같은 방식으로 인덱스 계산 -> 항상 같은 슬롯 찾음
        const index = this.hash(key);
        // [4-2] 해당 슬롯의 값 반환
        return this.table[index];
    }
}

// [5] HashTable 인스턴스 생성 -> 크기 10짜리 빈 배열 준비
const ht = new HashTable();

// [6] "name" -> hash() -> 인덱스 계산 -> 해당 슬롯에 "홍길동" 저장
ht.set("name", "홍길동");

// [7] "age" -> hash() -> 인덱스 계산 -> 해당 슬롯에 30 저장
ht.set("age", 30);

// [8] get()은 set()과 동일한 해시 함수 사용 -> 같은 인덱스 -> 값 반환
console.log("name:", ht.get("name")); // -> 홍길동
console.log("age:", ht.get("age"));   // -> 30


/* 키-값 쌍을 해시 인덱스로 빠르게 저장/조회하는 기본 해시 테이블 구현 (충돌 처리 없음)

1) JWT 사용자 세션 캐시
로그인한 유저 토큰을 `{ userId -> 토큰 }` 형태로 저장해두고 매번 DB 안 뒤져도 빠르게 인증 확인
2) API 요청 중복 방지
`{ 요청URL -> 마지막요청시각 }` 저장해서 같은 요청이 너무 짧은 간격으로 오면 차단 (rate limiting)
3) 게시판 조회수 카운터
`{ 게시글ID -> 조회수 }` 메모리에 올려두고 DB write는 주기적으로만 -> DB 부하 감소 */

Overwriting ex1.js


In [12]:
!node ex1.js

name: 홍길동
age: 30


In [14]:
%%writefile ex2.js

// < 충돌 해결 방식 1: 체이닝(Chaining) >

class ChainingHashTable {
    // [1] 생성자: 각 슬롯에 빈 배열([]) 생성 -> 충돌 시 같은 슬롯에 여러 값 연결 가능
    constructor(size = 5) {
        this.table = new Array(size);
        for (let i = 0; i < size; i++) {
            // [1-1] 슬롯마다 빈 배열 초기화 -> 나중에 충돌 항목들을 push로 쌓음
            this.table[i] = [];
        }
    }

    // [2] 해시 함수: 문자열 키 -> 0~4 범위 인덱스 반환 (ex1.js와 동일 방식)
    hash(key) {
        let sum = 0;
        for (let i = 0; i < key.length; i++) {
            // [2-1] 각 문자 아스키코드 누적 합산
            sum += key.charCodeAt(i);
        }
        // [2-2] % 테이블크기 -> 항상 유효한 인덱스 범위로 제한
        return sum % this.table.length;
    }

    // [3] set(): 충돌 발생해도 덮어쓰지 않고 배열에 push -> 체이닝 방식
    set(key, value) {
        // [3-1] 키 -> 인덱스 계산
        const index = this.hash(key);
        // [3-2] 해당 슬롯 배열에 {key, value} 객체 추가
        //       같은 인덱스에 이미 값이 있어도 덮어쓰지 않고 뒤에 이어붙임
        this.table[index].push({ key: key, value: value });
    }

    // [4] get(): 인덱스 찾은 뒤 버킷 안을 순회해서 키가 일치하는 항목 반환
    get(key) {
        // [4-1] 키 -> 인덱스 계산 (set()과 동일)
        const index = this.hash(key);
        // [4-2] 해당 슬롯의 배열(버킷) 가져옴
        //       버킷 = 같은 인덱스에 충돌된 항목들이 모여있는 배열
        const bucket = this.table[index];
        for (let i = 0; i < bucket.length; i++) {
            // [4-3] 버킷 안을 하나씩 순회하며 key가 일치하는 항목 탐색
            if (bucket[i].key === key) {
                return bucket[i].value; // [4-4] 일치하면 value 반환
            }
        }
        return null; // [4-5] 끝까지 못 찾으면 null 반환
    }

    // [5] print(): 테이블 전체 구조 출력 -> 어느 슬롯에 뭐가 들어있는지 확인용
    print() {
        console.log(this.table);
    }
}

// [6] 크기 5짜리 체이닝 해시 테이블 생성
const chainingTable = new ChainingHashTable(5);

// [7] 데이터 삽입 -> "cat"과 "dog"은 해시값 충돌 가능 (같은 슬롯에 체이닝됨)
chainingTable.set("cat", "고양이");
chainingTable.set("dog", "강아지");
chainingTable.set("bag", "가방");

// [8] 전체 테이블 출력 -> 충돌된 슬롯은 배열 안에 여러 객체가 쌓인 것 확인 가능
chainingTable.print();

// [9] 각 키로 값 조회 -> 버킷 순회해서 key 일치 항목 찾아 반환
console.log("cat 조회:", chainingTable.get("cat")); // -> 고양이
console.log("dog 조회:", chainingTable.get("dog")); // -> 강아지
console.log("bag 조회:", chainingTable.get("bag")); // -> 가방

Writing ex2.js


In [15]:
!node ex2.js

[
  [],
  [],
  [ { key: 'cat', value: '고양이' } ],
  [ { key: 'bag', value: '가방' } ],
  [ { key: 'dog', value: '강아지' } ]
]
cat 조회: 고양이
dog 조회: 강아지
bag 조회: 가방


In [16]:
%%writefile ex3.js

// < 충돌 해결 방식 2: 선형 조사(Linear Probing) >

class LinearProbingHashTable {
    // [1] 생성자: 크기 7짜리 배열 생성 -> 모든 슬롯을 null로 초기화
    //     체이닝(ex2)과 달리 슬롯 하나에 값 하나만 들어감
    constructor(size = 7) {
        this.table = new Array(size).fill(null);
    }

    // [2] 해시 함수: 문자열 키 -> 0~6 범위 인덱스 반환
    hash(key) {
        let sum = 0;
        for (let i = 0; i < key.length; i++) {
            // [2-1] 각 문자 아스키코드 누적 합산
            sum += key.charCodeAt(i);
        }
        // [2-2] % 테이블크기 -> 유효한 인덱스 범위로 제한
        return sum % this.table.length;
    }

    // [3] set(): 선형 탐사 방식으로 빈 슬롯 찾아 저장
    set(key, value) {
        // [3-1] 키 -> 인덱스 계산 (최초 목표 슬롯)
        let index = this.hash(key);
        // [3-2] 해당 슬롯이 null(빈칸)이 아니고 키도 다르면 -> 충돌 발생
        //       충돌 시 다음 슬롯으로 한 칸씩 이동 (선형 탐사)
        //       % this.table.length -> 끝에 도달하면 0번으로 순환
        while (this.table[index] !== null && this.table[index].key !== key) {
            index = (index + 1) % this.table.length;
        }
        // [3-3] null인 슬롯(빈칸) 또는 같은 키 슬롯 발견 -> 값 저장(덮어쓰기)
        this.table[index] = { key: key, value: value };
    }

    // [4] get(): 인덱스부터 시작해 키 일치 항목 찾을 때까지 선형 탐사
    get(key) {
        // [4-1] 키 -> 인덱스 계산
        let index = this.hash(key);
        // [4-2] 시작 인덱스 저장 -> 한 바퀴 다 돌았는지 확인용
        let startIndex = index;
        // [4-3] null 슬롯을 만나면 탐색 중단 (그 이후엔 저장된 적 없음)
        while (this.table[index] !== null) {
            // [4-4] 키 일치하면 값 반환
            if (this.table[index].key === key) {
                return this.table[index].value;
            }
            // [4-5] 키 불일치 -> 다음 슬롯으로 이동 (순환)
            index = (index + 1) % this.table.length;
            // [4-6] 시작 인덱스로 돌아오면 테이블 전체 탐색 완료 -> 종료
            if (index === startIndex) {
                break;
            }
        }
        return null; // [4-7] 끝까지 못 찾으면 null 반환
    }

    // [5] print(): 테이블 전체 출력 -> 각 슬롯 위치와 저장값 확인용
    print() {
        console.log(this.table);
    }
}

// [6] 크기 7짜리 선형 탐사 해시 테이블 생성
const probingTable = new LinearProbingHashTable(7);

// [7] 데이터 삽입 -> 충돌 시 체이닝(배열 추가) 대신 옆 슬롯으로 밀어냄
probingTable.set("cat", "고양이");
probingTable.set("dog", "강아지");
probingTable.set("bag", "가방");

// [8] 전체 테이블 출력 -> 충돌된 항목이 인접 슬롯에 분산된 것 확인 가능
probingTable.print();

// [9] 각 키로 값 조회 -> 선형 탐사로 저장 위치 추적해서 반환
console.log("cat 조회:", probingTable.get("cat")); // -> 고양이
console.log("dog 조회:", probingTable.get("dog")); // -> 강아지
console.log("bag 조회:", probingTable.get("bag")); // -> 가방

Writing ex3.js


In [17]:
!node ex3.js

[
  null,
  null,
  null,
  null,
  { key: 'cat', value: '고양이' },
  { key: 'bag', value: '가방' },
  { key: 'dog', value: '강아지' }
]
cat 조회: 고양이
dog 조회: 강아지
bag 조회: 가방


In [18]:
%%writefile ex4.js

// [1] Node.js 내장 암호화 모듈 불러옴 -> SHA-256 해시 생성에 사용
const crypto = require('crypto');

// [2] SHA-256 해시 함수: 문자열 입력 -> 64자리 16진수 문자열 반환
//     단방향 함수 -> 해시값으로 원본 복원 불가
function sha256(data) {
    return crypto.createHash('sha256') // [2-1] SHA-256 알고리즘 선택
                 .update(data)         // [2-2] 해시할 데이터 입력
                 .digest('hex');       // [2-3] 결과를 16진수 문자열로 반환
}

// [3] 솔트 적용 해시 함수: 비밀번호 + 솔트 합친 뒤 SHA-256 해시
//     솔트 = 무작위 문자열 -> 같은 비밀번호라도 다른 해시값 생성
//     레인보우 테이블 공격(미리 계산된 해시 목록으로 역추적) 방어용
function hashWithSalt(password, salt) {
    return sha256(password + salt); // [3-1] "1234ABC123" 처럼 합쳐서 해시
}

const password = "1234";
const salt1 = "ABC123";
const salt2 = "XYZ999";

// [4] 같은 비밀번호 + 다른 솔트 -> 완전히 다른 해시값 생성
const hash1 = hashWithSalt(password, salt1); // "1234" + "ABC123" -> 해시
const hash2 = hashWithSalt(password, salt2); // "1234" + "XYZ999" -> 해시

// [5] 같은 비밀번호 + 같은 솔트 -> 항상 동일한 해시값 생성 (결정론적)
const hash3 = hashWithSalt(password, salt1); // hash1과 입력 동일 -> 결과도 동일

console.log("비밀번호:", password);
console.log("salt1 사용:", hash1);
console.log("salt2 사용:", hash2);          // hash1과 다름 -> 솔트가 다르기 때문
console.log("같은 salt1 다시 사용:", hash3); // hash1과 동일 -> 입력이 같기 때문

// [6] 검증 출력
// false -> 솔트가 다르면 같은 비밀번호도 다른 해시값 나옴 (솔트의 핵심 역할)
console.log("hash1 === hash2:", hash1 === hash2);
// true  -> 솔트까지 동일하면 해시값도 항상 같음 (로그인 검증 원리)
console.log("hash1 === hash3:", hash1 === hash3);

Writing ex4.js


In [19]:
!node ex4.js

비밀번호: 1234
salt1 사용: 9a4ac681f778d16757e95202da9caf6039b9158f801fe228b3a9a75108712211
salt2 사용: 1f7bb80246f64b768bf3fe92ca4afa633bb37362534cab6f0b5464daaa3a79e5
같은 salt1 다시 사용: 9a4ac681f778d16757e95202da9caf6039b9158f801fe228b3a9a75108712211
hash1 === hash2: false
hash1 === hash3: true


In [20]:
%%writefile ex5.js

// [1] Node.js 내장 암호화 모듈 불러옴
const crypto = require('crypto');

// [2] SHA-256 해시 함수: 문자열 -> 64자리 16진수 해시값 반환
//     입력이 1글자라도 바뀌면 해시값이 완전히 달라짐 -> 무결성 검증에 활용
function sha256(data) {
    return crypto.createHash('sha256').update(data).digest('hex');
}

// [3] 데이터 전송: 원본 데이터 + 해시값을 묶어서 패킷으로 반환
//     해시값 = 데이터의 "지문" -> 수신 측에서 비교용으로 사용
function sendData(data) {
    const hash = sha256(data);   // [3-1] 원본 데이터의 해시값 생성
    return {
        payload: data,           // [3-2] 원본 데이터
        hash: hash               // [3-3] 원본 해시값 (검증 기준)
    };
}

// [4] 데이터 수신: 받은 payload를 다시 해시화 -> 기존 hash와 비교
//     두 값이 같으면 전송 중 데이터가 변조되지 않은 것
function receiveData(packet) {
    // [4-1] 수신된 payload로 해시 재계산
    const newHash = sha256(packet.payload);
    // [4-2] 재계산한 해시 vs 전송 시 첨부된 해시 비교
    if (newHash === packet.hash) {
        // [4-3] 일치 -> payload가 중간에 바뀌지 않음
        console.log("무결성 검증 성공: 데이터가 변경되지 않음");
    } else {
        // [4-4] 불일치 -> payload가 변조됨
        //       hash는 원본 기준이라 변조된 payload와 맞지 않음
        console.log("무결성 검증 실패: 데이터가 변경됨");
    }
}

// [5] 정상 케이스: payload와 hash가 모두 원본 -> 검증 성공
const packet1 = sendData("송금금액:10000원");
receiveData(packet1); // -> 무결성 검증 성공

// [6] 변조 케이스: hash는 "10000원" 기준인데 payload만 "90000원"으로 교체
//     hash를 같이 바꾸지 않으면 검증에서 걸림 -> 실제 금융 시스템 보안 원리
const packet2 = sendData("송금금액:10000원");
packet2.payload = "송금금액:90000원"; // [6-1] payload만 몰래 변조
receiveData(packet2); // -> 무결성 검증 실패

Writing ex5.js


In [21]:
!node ex5.js

무결성 검증 성공: 데이터가 변경되지 않음
무결성 검증 실패: 데이터가 변경됨


In [22]:
%%writefile ex6.js

const crypto = require('crypto');

function sha256(data) {
    return crypto.createHash('sha256').update(data).digest('hex');
}

// [1] 비교할 파일 5개 준비
//     A와 B는 내용 동일 -> 해시값도 동일해야 함 (무결성 검증 핵심)
//     C, D, E는 내용 다름 -> 해시값도 완전히 달라짐
const fileA = "비트코인(Bitcoin)은 2009년 사토시 나카모토가 만든 최초의 암호화폐입니다.";
const fileB = "비트코인(Bitcoin)은 2009년 사토시 나카모토가 만든 최초의 암호화폐입니다."; // A와 동일
const fileC = "비트코인(Bitcoin)은 2009년 사토시 나카모토가 만든 최초의 암호화폐입니다!"; // 느낌표 1개 차이
const fileD = "이더리움(Ethereum)은 스마트 컨트랙트 기능을 지원하는 블록체인 플랫폼입니다.";
const fileE = "해시 함수는 임의의 데이터를 고정된 길이의 값으로 변환하는 단방향 함수입니다.";

// [2] 각 파일의 SHA-256 해시값 계산
const hashA = sha256(fileA);
const hashB = sha256(fileB);
const hashC = sha256(fileC);
const hashD = sha256(fileD);
const hashE = sha256(fileE);

// [3] 해시값 출력 -> 내용이 조금이라도 다르면 해시값이 완전히 달라지는 것 확인
console.log("=== 해시값 확인 ===");
console.log("A:", hashA);
console.log("B:", hashB); // A와 동일한 해시값
console.log("C:", hashC); // 느낌표 1개 차이인데 해시값은 완전히 다름
console.log("D:", hashD);
console.log("E:", hashE);

// [4] 동일 여부 비교
console.log("\n=== 무결성 비교 결과 ===");
// true  -> 내용 완전 동일 -> 파일 복사/전송 중 변조 없음
console.log("A vs B 동일:", hashA === hashB);
// false -> 느낌표 1개 차이 -> 해시값은 완전히 달라짐 (avalanche effect)
console.log("A vs C 동일:", hashA === hashC);
// false -> 내용 다름
console.log("A vs D 동일:", hashA === hashD);
// false -> 내용 다름
console.log("A vs E 동일:", hashA === hashE);
// false -> 내용 다름
console.log("D vs E 동일:", hashD === hashE);

/* ── 실무 활용 예시 ─────────────────────────────────────
 *
 *  1) 회원가입 비밀번호 저장
 *     "1234" 그대로 DB에 저장하면 해킹 시 바로 노출
 *     -> 해시값으로 변환해서 저장 -> 원본 복원 불가
 *
 *  2) 로그인 검증
 *     입력한 비밀번호를 해시화 -> DB에 저장된 해시값과 비교
 *     -> 일치하면 로그인 성공 (원본 비밀번호는 서버도 모름)
 *
 *  3) 게시글/댓글 변조 감지
 *     글 저장 시 해시값도 같이 저장
 *     -> 나중에 내용 해시값 다시 계산해서 비교
 *     -> 다르면 누군가 DB를 직접 수정한 것
 *
 *  4) 파일 업로드 중복 방지
 *     업로드된 이미지의 해시값 계산
 *     -> 이미 같은 해시값이 있으면 동일 파일로 판단
 *     -> 저장 안 하고 기존 파일 재사용 (용량 절약)
 * ────────────────────────────────────────────────────── */

Writing ex6.js


In [23]:
!node ex6.js

=== 해시값 확인 ===
A: 05d407421d2b95eade97d5192fa53e91672a6c68d26ada21cdd7cc3ac65420f6
B: 05d407421d2b95eade97d5192fa53e91672a6c68d26ada21cdd7cc3ac65420f6
C: c2dff29b5c72e44028c92db819a10d88fa389421b48cdb2115473384479fdead
D: d0958b6e3979761d9da205074bbefee848ecae5ea28bac1da631105d4c83b104
E: b29755f0cdc9f7b858c6297d6c14bb6887eac7ae9fff625d0d319f96287ff5ea

=== 무결성 비교 결과 ===
A vs B 동일: true
A vs C 동일: false
A vs D 동일: false
A vs E 동일: false
D vs E 동일: false


In [26]:
%%writefile ex7.js

class ChainingHashTable {
    // [1] 생성자: 각 슬롯에 빈 배열 초기화 -> 충돌 시 같은 슬롯에 여러 값 연결
    constructor(size = 5) {
        this.table = new Array(size);
        for (let i = 0; i < size; i++) {
            this.table[i] = []; // [1-1] 슬롯마다 빈 배열 -> 체이닝 준비
        }
    }

    // [2] 해시 함수: 문자열 키 -> 인덱스 변환
    hash(key) {
        let sum = 0;
        for (let i = 0; i < key.length; i++) {
            sum += key.charCodeAt(i);
        }
        return sum % this.table.length;
    }

    // [3] set(): 키(긴 텍스트) -> 해시 인덱스 -> 해당 버킷에 메타데이터 저장
    set(key, value) {
        const index = this.hash(key);
        this.table[index].push({ key: key, value: value });
    }

    // [4] get(): 키로 인덱스 찾고 버킷 순회 -> 키 일치 항목의 value 반환
    get(key) {
        const index = this.hash(key);
        const bucket = this.table[index];
        for (let i = 0; i < bucket.length; i++) {
            if (bucket[i].key === key) {
                return bucket[i].value;
            }
        }
        return null;
    }

    print() {
        console.log(this.table);
    }
}

// [5] 테이블 크기 10으로 생성
const fileTable = new ChainingHashTable(10);

// [6] 문서 데이터 5개 준비 -> 내용 자체를 키로 사용
const bigData1 = "블록체인(Blockchain)은 데이터를 블록 단위로 연결해 분산 저장하는 기술로, 위변조가 사실상 불가능하다.";
const bigData2 = "인공지능(AI)은 머신러닝과 딥러닝을 기반으로 인간의 학습, 추론, 판단 능력을 컴퓨터로 구현한 기술이다.";
const bigData3 = "REST API는 HTTP 메서드(GET, POST, PUT, DELETE)를 사용해 서버 자원을 주고받는 통신 방식이다.";
const bigData4 = "JWT(JSON Web Token)는 사용자 인증 정보를 JSON 형태로 인코딩해 서버-클라이언트 간 전달하는 토큰이다.";
const bigData5 = "해시 함수는 임의 길이의 데이터를 고정 길이의 값으로 변환하며, 동일 입력은 항상 동일 출력을 보장한다.";

// [7] 각 문서 내용 -> 해시 -> 인덱스 -> 메타데이터 저장
fileTable.set(bigData1, { author: "김블록", date: "2024-01-15", type: "PDF",  category: "블록체인" });
fileTable.set(bigData2, { author: "이AI",   date: "2024-02-20", type: "DOCX", category: "인공지능" });
fileTable.set(bigData3, { author: "박서버", date: "2024-03-05", type: "PDF",  category: "백엔드"   });
fileTable.set(bigData4, { author: "최보안", date: "2024-03-18", type: "MD",   category: "보안"     });
fileTable.set(bigData5, { author: "정해시", date: "2024-03-31", type: "PDF",  category: "자료구조" });

// [8] 내용 그대로 넣으면 저장 위치 몰라도 즉시 조회
console.log("--- 데이터 기반 즉시 조회 ---");
console.log("bigData1 조회:", fileTable.get(bigData1)); // -> { author: "김블록", ... }
console.log("bigData2 조회:", fileTable.get(bigData2)); // -> { author: "이AI",   ... }
console.log("bigData3 조회:", fileTable.get(bigData3)); // -> { author: "박서버", ... }
console.log("bigData4 조회:", fileTable.get(bigData4)); // -> { author: "최보안", ... }
console.log("bigData5 조회:", fileTable.get(bigData5)); // -> { author: "정해시", ... }

// [9] 내용이 1글자라도 다르면 해시값 자체가 달라짐 -> null 반환
//     데이터 변조 감지 효과도 동시에 얻을 수 있음
console.log("\n--- 변조된 내용 조회 시도 ---");
console.log("변조 데이터 조회:", fileTable.get("블록체인은 위변조가 불가능하다.")); // -> null

/* ── 실무 활용 예시 ─────────────────────────────────────
 *
 *  1) 문서 관리 시스템
 *     계약서/보고서 내용 자체를 키로 저장
 *     -> 내용만 알면 작성자/날짜/형식 즉시 조회
 *     -> 내용이 바뀌면 조회 안 됨 -> 변조 자동 감지
 *
 *  2) 이미지 중복 업로드 방지
 *     업로드된 이미지 데이터를 해시화 -> 키로 저장
 *     -> 같은 이미지 또 올리면 이미 있는 파일로 판단
 *     -> 저장 안 하고 기존 파일 재사용 (서버 용량 절약)
 *
 *  3) 게시글 캐싱
 *     게시글 내용을 키, 조회수/댓글수를 값으로 저장
 *     -> DB 매번 안 뒤지고 메모리에서 바로 꺼냄
 *     -> 내용 수정되면 키가 달라져서 자동으로 캐시 무효화
 * ────────────────────────────────────────────────────── */

Overwriting ex7.js


In [25]:
!node ex7.js

--- 데이터 기반 즉시 조회 ---
bigData1 조회: { author: '김블록', date: '2024-01-15', type: 'PDF', category: '블록체인' }
bigData2 조회: { author: '이AI', date: '2024-02-20', type: 'DOCX', category: '인공지능' }
bigData3 조회: { author: '박서버', date: '2024-03-05', type: 'PDF', category: '백엔드' }
bigData4 조회: { author: '최보안', date: '2024-03-18', type: 'MD', category: '보안' }
bigData5 조회: { author: '정해시', date: '2024-03-31', type: 'PDF', category: '자료구조' }

--- 변조된 내용 조회 시도 ---
변조 데이터 조회: null


In [27]:
%%writefile ex8.js

const crypto = require('crypto');

// [1] SHA-256 해시 함수
function sha256(data) {
    return crypto.createHash('sha256').update(data).digest('hex');
}

// [2] 솔트 적용 해시 함수: 비밀번호 + 솔트 합쳐서 해시화
//     같은 비밀번호라도 솔트가 다르면 해시값이 완전히 달라짐
function hashWithSalt(password, salt) {
    return sha256(password + salt);
}

// [3] 회원가입 시점: 솔트 생성 + 비밀번호 해시화해서 저장
//     실제 서비스에서는 솔트를 crypto.randomBytes()로 랜덤 생성
const user = {
    id: "user1",
    salt: "ABC123",                              // [3-1] 유저마다 고유한 솔트 부여
    passwordHash: hashWithSalt("1234", "ABC123") // [3-2] 원본 비밀번호는 저장 안 함
};

console.log("=== 저장된 유저 정보 ===");
console.log("id:", user.id);
console.log("salt:", user.salt);
console.log("passwordHash:", user.passwordHash); // 해시값만 저장됨
console.log("");

// [4] 로그인 함수: 입력 비밀번호 + 저장된 솔트 -> 해시화 -> 저장값과 비교
function login(inputPw) {
    // [4-1] 입력 비밀번호에 저장된 솔트 붙여서 동일한 방식으로 해시화
    const inputHash = hashWithSalt(inputPw, user.salt);
    // [4-2] 해시값끼리 비교 -> 원본 비밀번호는 서버도 모름
    if (inputHash === user.passwordHash) {
        console.log(`[${inputPw}] 입력 -> 로그인 성공`);
    } else {
        console.log(`[${inputPw}] 입력 -> 로그인 실패`);
    }
}

// [5] 로그인 시도
console.log("=== 로그인 시도 ===");
login("1234"); // -> 성공: 해시값 일치
login("1111"); // -> 실패: 해시값 불일치

// [6] 솔트 효과 확인: 같은 비밀번호라도 솔트가 다르면 해시값 달라짐
console.log("\n=== 솔트 효과 확인 ===");
const hash1 = hashWithSalt("1234", "ABC123"); // user의 솔트
const hash2 = hashWithSalt("1234", "XYZ999"); // 다른 솔트
const hash3 = hashWithSalt("1234", "ABC123"); // user의 솔트 재사용
console.log("같은 솔트(hash1 === hash3):", hash1 === hash3); // -> true
console.log("다른 솔트(hash1 === hash2):", hash1 === hash2); // -> false
// 레인보우 테이블 공격 방어: "1234"의 해시값을 미리 알고 있어도
// 솔트가 섞이면 그 해시값과 일치하지 않음 -> 역추적 불가

Writing ex8.js


In [29]:
!node ex8.js

=== 저장된 유저 정보 ===
id: user1
salt: ABC123
passwordHash: 9a4ac681f778d16757e95202da9caf6039b9158f801fe228b3a9a75108712211

=== 로그인 시도 ===
[1234] 입력 -> 로그인 성공
[1111] 입력 -> 로그인 실패

=== 솔트 효과 확인 ===
같은 솔트(hash1 === hash3): true
다른 솔트(hash1 === hash2): false
